# A1: Logic and lambda calculus

**Simon Dobnik and Robin Cooper**

The lab is an exploration and learning exercise to be done in a group and also in discussion with the teachers and other students.

Before starting, please read the instructions on how to work in groups on Canvas.

Write all your answers and the code in the appropriate boxes below.

**On using generative AI for this assignment:** Generative AI has been trained to generate logic expressions. There is floruishing ongoing research how to bridge formal and distributed models to bring improve to the latter known as *neuro-symbolic AI*. Since generative AI can give you direct solutions to this assignment (which may be correct or incorrect), defeats the purpose of doingthe assignment which is to familiarise yourself with writing logic-based semantic grammars. Using generative AI is thus identical to copying an asnwer from someone else. For this reason, **generative AI should not be used** to complete this assignment.

**Running the notebook:** The notebook requires a few dependencies, most notably NLTK which is widely available. However, to draw graphical trees a few other libraries are required but which some students found hard to install on their own computers in the previous years, depedning on what configurations they were using. However, drawing trees graphically is not necessary to work on the assignment as you can also use the bracketing notation in case you cannot get them work. Atternatively, you can run the notebook on the mlt server trhough an ssh tunnel where these dependencies should be available for install.

We recommend that you create a virtual environment (either with virtualenv or conda), install Jupyter Lab in it and all dependencies used in the assignment. To run Jupyter Lab later within your environment, run the following command:

```python -m ipykernel install --user --name=my-virtualenv-name```,

where you replace `my-virtualenv-name` with the name of your created environment.
Once in Jupyter, choose the kernel with the name of your environment. You can do it by either (i) using the drop-down menu in the top right corner or (ii) going to the top menu -> Kernel -> Change Kernel.

**Getting help:** We encourage you to use Canvas discussions to post questions and interact with teachers and also each other. Provide youseful tips, but of course do not reveal the exact answer across the groups as each group should should work out their own solutions. Remember that in most cases there is also not a single correct answer.

In [13]:
!sudo apt-get update

Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app


## Translating English to logic and evaluating logic in a model

In [1]:
# This task needs NLTK and Jupyter Lab (IPython package).
import nltk
from utils import display_latex, display_translation, display_tree, display, Markdown
read_expr = nltk.sem.Expression.fromstring

### 1. Propositional logic
Translate the following sentences into **propositional logic** and verify that they parse with Expression.fromstring() (`read_expr` variable in the cell above). Provide a key which shows how the propositional variables in your translation correspond to expressions of English. Briefly discuss any difficulties you encounter. (By difficulties we mean cases where the semantics of English expressions cannot be expressed to the same degree by the semantics of your logic representations, i.e. they do not mean the same). **[5 + 1 marks]**

---
### **Key**:
- P = Alex plays piano 
- S = Alex is smart 
- M = Alex is musical 
- LS = Lydia is smart 
- LH = Lydia is happy 
- LM = Lydia is musical
- G = Georg plays the piano  
- MG = Georg is musical
---

In [2]:
propositions = {
    "If Alex plays the piano, she is smart.":
    read_expr('P -> S'),
    
    "Alex is both smart and musical.":
    read_expr('S & M'),
    
    "If Alex is not smart, Lydia is not happy.":
    read_expr('-S -> -LH'),
    
    "If Alex or George plays the piano, they are musical.":
    read_expr('(P -> M) & (G -> MG)'),
    
    "George plays the piano.":
    read_expr('G'),
}

for text, semrep in propositions.items():
    display_translation(text, semrep)

"If Alex plays the piano, she is smart.": $(P\ \rightarrow\ S)$

"Alex is both smart and musical.": $(S\ \land\ M)$

"If Alex is not smart, Lydia is not happy.": $(-S\ \rightarrow\ -LH)$

"If Alex or George plays the piano, they are musical.": $((P\ \rightarrow\ M)\ \land\ (G\ \rightarrow\ MG))$

"George plays the piano.": $G$

*Difficulties encountered:*

Sentences 1, 2, 3 and 5 are straightforward and can be expressed well in 
propositional logic, but sentence 4 is tricky.

At first we wrote `(P | G) -> (M & MG)`, but this means both Alex and George 
must be musical regardless of who plays piano. For example, if only Alex plays 
(`P & -G`), the formula still incorrectly requires George to be musical.

The corrected formula `(P -> M) & (G -> MG)` instead says: whoever plays piano 
must be musical. If either of them plays but is not musical, the formula returns False.

- **Pronoun resolution / "they"**: "they" is a plural pronoun but propositional 
logic has no mechanism for anaphora or reference tracking; we had to manually 
split it into two implications.

- **Inclusive vs. exclusive "or"**: English "or" in sentence 4 is ambiguous; 
`|` is always inclusive or, which may not match the intended reading.

### 2. Valuation of Propositional logic

Imagine that we observe a world where 
- (i) Alex does not play the piano,
- (ii) Alex and Lydia are smart and musical,
- (iii) George is not musical,
- (iv) Lydia is happy,
- (v) George plays the piano. 

Translate this informal description of the world into a model by appropriately defining an evaluation function and evaluate the formulae from Question 1 in this model. Briefly comment the answers you get. **[5 + 1 marks]**.

In [3]:
# answers here
def build_and_evaluate_world(valuation, formulas):
    dom = set() # domain (which we don't have for PPL but we have to define it)
    g = nltk.Assignment(dom)
    m = nltk.Model(dom, nltk.Valuation(valuation))
    for f in formulas:
        print(f"{f} : {m.evaluate(f, g)}")

valuation = [
    ('P', False),
    ('S', True),
    ('M', True),
    ('LS', True),
    ('LM', True),
    ('LH', True),
    ('MG', False),
    ('G', True)
]

formulas = ['P -> S', 'S & M', '-S -> -LH', '(P -> M) & (G -> MG)', 'G']

build_and_evaluate_world(valuation, formulas)

P -> S : True
S & M : True
-S -> -LH : True
(P -> M) & (G -> MG) : False
G : True


*Comments:*

> In this world:
> - P = False (Alex does not play piano)
> - S = True (Alex is smart)
> - M = True (Alex is musical)
> - LS = True (Lydia is smart)
> - LM = True (Lydia is musical)
> - LH = True (Lydia is happy)
> - G = True (George plays piano)
> - MG = False (George is not musical)

---

- **P -> S**: P is False, so the implication is **True** (false antecedent)

- **S & M**: S and M are both True, so the conjunction is **True**

- **-S -> -LH**: -S is False, so the implication is **True** (false antecedent)

- **(P -> M) & (G -> MG)**: P is False so (P -> M) is True; G is True and MG 
is False so (G -> MG) is False; True & False = **False**

- **G**: G is True, so the result is **True**

### 3. Predicate logic *without quantifiers*

Translate the following sentences into predicate-argument formulae of First Order Logic and verify that they parse with `Expression.fromstring()`. Briefly discuss any difficulties you encounter. **[4 + 1 marks]**

In [4]:
sentences1 = {
    "Lydia likes George but Lydia doesn't like Alex": 
    read_expr('like(lydia, george) & - like(lydia, alex)'),
    
    "Lydia likes herself and so does George":
    read_expr('like(lydia, lydia) & like(george, lydia)'),
    
    "Charlie is an English pianist who plays a sonata":
    read_expr('english(charlie) & pianist(charlie) & play(charlie, sonata)'),
    
    "Lydia and George admire each other":
    read_expr('admire(lydia, george) & admire(george, lydia)'),
}

for text, semrep in sentences1.items():
    display_translation(text, semrep)


"Lydia likes George but Lydia doesn't like Alex": $(like(lydia,george)\ \land\ -like(lydia,alex))$

"Lydia likes herself and so does George": $(like(lydia,lydia)\ \land\ like(george,lydia))$

"Charlie is an English pianist who plays a sonata": $(english(charlie)\ \land\ pianist(charlie)\ \land\ play(charlie,sonata))$

"Lydia and George admire each other": $(admire(lydia,george)\ \land\ admire(george,lydia))$

*Difficulties encountered:*  
We were unsure of how to interpret the second sentence: 
> Lydia likes herself, but does George like Lydia or does he like himself?   

we decided to interpret it as Lydia liking herself and George also liking Lydia, since the pronoun "herself" is used. If the meaning was George liking himself, the pronoung "himself" should be used somewhere.

### 4. First order logic with quantifiers

Translate the following sentences into quantified formulas of First Order Logic and verify that they parse with `Expression.fromstring()`. Briefly discuss any difficulties you encounter. **[4 + 1 marks]**

In [5]:
sentences2 = {
    "Charlie knows a woman who likes George":
    read_expr('exists x.(woman(x) & know(charlie, x) & like(x, george))'),
    
    "George admires everybody and Lydia admires nobody":
    read_expr('all x.(admire(george, x)) & -exists x.(admire(lydia, x))'),

    "Nobody admires everybody":
    read_expr('-exists x. all y. (admire(x, y))'),
    
    "Exactly one musician plays everything Alex wrote":
    read_expr('exists x. (musician(x) & all y. (write(alex,y) -> play(x,y)) & all z. ((musician(z) & all y. (write(alex,y) -> play(z,y))) -> z = x))'),
}

for text, semrep in sentences2.items():
    display_translation(text, semrep)

"Charlie knows a woman who likes George": $\exists\ x.(woman(x)\ \land\ know(charlie,x)\ \land\ like(x,george))$

"George admires everybody and Lydia admires nobody": $(\forall\ x.admire(george,x)\ \land\ -\exists\ x.admire(lydia,x))$

"Nobody admires everybody": $-\exists\ x.\forall\ y.admire(x,y)$

"Exactly one musician plays everything Alex wrote": $\exists\ x.(musician(x)\ \land\ \forall\ y.(write(alex,y)\ \rightarrow\ play(x,y))\ \land\ \forall\ z.((musician(z)\ \land\ \forall\ y.(write(alex,y)\ \rightarrow\ play(z,y)))\ \rightarrow\ (z\ =\ x)))$

*Difficulties encountered:*

- **Quantifier scope ambiguity**: English doesn't mark scope explicitly and FOL 
requires precise bracketing and small errors completely change the meaning. 
For example, our first attempt had a free variable bug:  
`exists x.(woman(x) & know(charlie, x)) & like(x, george)`

- **Natural language quantifiers**: "nobody" and "everybody" must be decomposed 
into `exists`/`all` with negation, and the mapping is not always straightforward.

- **Exactly one**: We spent so much time finding out how to say exactly one musician in sentence 4. And we found this way:
  - Existence: `∃x P(x)` —> "there exists an x with property P"
  - Uniqueness: `∀z(P(z) → z=x)` —> "any other individual with P must equal x"
  - Combined: `∃x(P(x) ∧ ∀z(P(z) → z=x))`

### 5. Valuation of first order logic

We observe a world with entities Lydia, George, Alex, Charlie and Bertie, sonata, etude, prelude, waltz, scherzo.

1. Lydia likes Lydia, George, Alex and Charlie. George likes Lydia, Bertie and George. Alex likes Alex. Charlie likes Lydia, George, Alex, Charlie and Bertie. Bertie likes Alex.
2. Lydia, George, Alex, Charlie and Bertie are English.
3. Charlie and Bertie are pianists.
4. Charlie plays a sonata, an etude and a waltz. Bertie plays a waltz and a scherzo. Lydia plays an etude, a prelude and a waltz.
5. Lydia admires Lydia, Charlie and Bertie. George admires Lydia, George, Alex, Charlie and Bertie. Alex admires Lydia, Alex and Bertie. Charlie admires George and Bertie. Bertie admires Lydia, George, Alex, Charlie and Bertie.
6. Lydia knows Lydia, George, Alex, Charlie and Bertie. George knows Lydia, George and Bertie. Alex knows Lydia, Alex and Bertie. Charlie knows George, Charlie and Bertie. Bertie knows Lydia, George, Alex, Charlie and Bertie.
7. Lydia, Alex and Charlie are women.
8. George and Bertie are men.
9. Alex wrote a sonata, an etude an a waltz.
10. Lydia, Alex, Charlie and Bertie are musicians.

Translate this informal description of the world into a model and evaluate the formulae from Questions 3 and 4 in this model. Briefly comment on the answers you get **[3 + 2 marks]**.

In [6]:
entities = set(['l','g','a','c','b','s','e','p','w','sch'])

assign = """
    lydia => l
    george => g
    alex => a
    charlie => c
    bartie => b
    sonata => s
    etude => e
    prelude => p
    waltz => w
    scherzo => sch
    like => {(l,l), (l,g), (l,a), (l,c), (g,l), (g,b), (g,g), (a,a), (c,l), (c,g), (c,a), (c,c), (c,b), (b,a)}
    english => {l, g, a, c, b}
    pianist => {c, b}
    play => {(c, s), (c, e), (c, w), (b, w), (b, sch), (l, e), (l, p), (l, w)}
    admire => {(l,l), (l,c), (l,b), (g,l), (g,g), (g,a), (g,b), (g,c), (g,b), (a,l), (a,a), (a,b), (c,g), (c,b), (b,l), (b,g), (b,a), (b,c), (b,b)}
    know => {(l,l), (l,g), (l,a), (l,c), (l, b), (g,l), (g,g), (g,b), (a,l), (a,a), (a,b), (c,g), (c,c), (c,b), (b,l), (b,g), (b,a), (b,c), (b,b)}
    woman => {l, a, c}
    man => {g, b}
    write => {(a,s), (a,e), (a,w)}
    musician => {l, a, c, b}
"""


val2 = nltk.Valuation.fromstring(assign)

g2 = nltk.Assignment(entities)
m2 = nltk.Model(entities, val2)

# sentences from question 3
for text, semrep in sentences1.items():
    print(m2.evaluate(str(semrep), g2))
    display_latex(semrep)
    display(Markdown('----'))

# sentences from question 4
for text, semrep in sentences2.items():
    print(m2.evaluate(str(semrep), g2))
    display_latex(semrep)
    display(Markdown('----'))


False


$(like(lydia,george)\ \land\ -like(lydia,alex))$

----

True


$(like(lydia,lydia)\ \land\ like(george,lydia))$

...

$-\exists\ x.\forall\ y.admire(x,y)$

----

True


$\exists\ x.(musician(x)\ \land\ \forall\ y.(write(alex,y)\ \rightarrow\ play(x,y))\ \land\ \forall\ z.((musician(z)\ \land\ \forall\ y.(write(alex,y)\ \rightarrow\ play(z,y)))\ \rightarrow\ (z\ =\ x)))$

----

*Comments on the answers:*  

All the formulae are evaluated correctly according to the world observed here, except for one.

> "Nobody admires everybody" (Question 4, sentence 3) parses as True, even though Bertie and George admire everybody.

This is due to the musical pieces being included in the entities. Therefore the program thinks that these are part of "everybody" too, since it does not know the difference between people and musical pieces. But nobody admires all the musical pieces too.

We tested this by including `(g, s)`, `(g, e)`, `(g, p)`, `(g, w)`, `(g, sch)` in `admire => {}` so that there was someone (George) who admired all the people and musical pieces. As a result, the program returned false like it was supposed to.

A solution to that problem is to limit the entities that are included in "everyone" to every entity that is a person (in this case men and women) and use a more specific formula in Question 4:

```
read_expr('-exists x. all y. ((man(x) | woman(x)) & (man(y) | woman(y)) -> admire(x,y))')
```


## Lambda calculus

In [7]:
from nltk.grammar import FeatureGrammar

### 6. Function application and $\beta$-reduction
In the following examples some code has been deleted and replaced with `<????>`. What has been deleted? Verify that your answer is correct. **[4 marks]**

In [8]:
e1 = read_expr(r'\x.(like(x, rob))')
e2 = read_expr(r'pip')
e3 = nltk.sem.ApplicationExpression(e1,e2) 
display_latex(e3.simplify())
# with result like(pip,rob).
display_latex(read_expr(r"like(pip,rob)"))

e1 = read_expr(r'\P.P(pip)')
e2 = read_expr(r'\x.play(x,scherzo)') 
e3 = nltk.sem.ApplicationExpression(e1,e2)
display_latex(e3.simplify())
# with result play(pip,scherzo).
display_latex(read_expr(r"play(pip,scherzo)"))

e1 = read_expr(r'\P.exists x.(woman(x) & P(x))')
e2 = read_expr(r'\x.play(x,etude)') 
e3 = nltk.sem.ApplicationExpression(e1,e2) 
display_latex(e3.simplify())
# with result exists x.(woman(x) & play(x,etude)).
display_latex(read_expr(r"exists x.(woman(x) & play(x,etude))"))

e1 = read_expr(r'\X x.X(\y.like(x,y))')
e2 = read_expr(r'\P.all x. (musician(x) -> P(x))') 
e3 = nltk.sem.ApplicationExpression(e1,e2) 
display_latex(e3.simplify())
# with result \x.all z2.(musician(z2) -> like(x,z2)).
display_latex(read_expr(r"\x.all z2.(musician(z2) -> like(x,z2))"))

$like(pip,rob)$

$like(pip,rob)$

$play(pip,scherzo)$

$play(pip,scherzo)$

$\exists\ x.(woman(x)\ \land\ play(x,etude))$

$\exists\ x.(woman(x)\ \land\ play(x,etude))$

$\lambda\ x.\forall\ z_{1}.(musician(z_{1})\ \rightarrow\ like(x,z_{1}))$

$\lambda\ x.\forall\ z_{2}.(musician(z_{2})\ \rightarrow\ like(x,z_{2}))$

### 7. Extending the grammar

Extend the grammar simple_sem.fcfg that comes with NLTK `(~/nltk_data/grammars/book_grammars/)` so that it will cover the following sentences:

- no man gives a bone to a dog **[4 marks]**
- no man gives a bone to the dog **[4 marks]**
- a boy and a girl chased every dog **[2 marks]**
- every dog chased a boy and a girl **[2 marks]**
- a brown cat chases a white dog **[4 marks]**

The last example includes adjectives. Several different kinds of adjectives are discussed in the literature [(cf. Kennedy, 2012)](http://semantics.uchicago.edu/kennedy/docs/routledge.pdf). In this example we have an intersective adjective. The denotiation we want for "brown cat" is a a set that we get by intersecting the set of individuals that are brown and the set of individuals that are cats.

C. Kennedy. Adjectives. In G. Russell, editor, The Routledge Companion to Philosophy of Language, chapter 3.3, pages 328–341. Routledge, 2012.

The original grammar is included in the code below as a string.

In [9]:
fcfg_string_orginal = r"""
% start S
############################
# Grammar Rules
#############################

S[SEM = <?subj(?vp)>] -> NP[NUM=?n,SEM=?subj] VP[NUM=?n,SEM=?vp]

NP[NUM=?n,SEM=<?det(?nom)> ] -> Det[NUM=?n,SEM=?det]  Nom[NUM=?n,SEM=?nom]
NP[LOC=?l,NUM=?n,SEM=?np] -> PropN[LOC=?l,NUM=?n,SEM=?np]

Nom[NUM=?n,SEM=?nom] -> N[NUM=?n,SEM=?nom]

VP[NUM=?n,SEM=?v] -> IV[NUM=?n,SEM=?v]
VP[NUM=?n,SEM=<?v(?obj)>] -> TV[NUM=?n,SEM=?v] NP[SEM=?obj]
VP[NUM=?n,SEM=<?v(?obj,?pp)>] -> DTV[NUM=?n,SEM=?v] NP[SEM=?obj] PP[+TO,SEM=?pp]

PP[+TO, SEM=?np] -> P[+TO] NP[SEM=?np]

#############################
# Lexical Rules
#############################

PropN[-LOC,NUM=sg,SEM=<\P.P(angus)>] -> 'Angus'
PropN[-LOC,NUM=sg,SEM=<\P.P(cyril)>] -> 'Cyril'
PropN[-LOC,NUM=sg,SEM=<\P.P(irene)>] -> 'Irene'
 
Det[NUM=sg,SEM=<\P Q.all x.(P(x) -> Q(x))>] -> 'every'
Det[NUM=pl,SEM=<\P Q.all x.(P(x) -> Q(x))>] -> 'all'
Det[SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'some'
Det[NUM=sg,SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'a'
Det[NUM=sg,SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'an'

N[NUM=sg,SEM=<\x.man(x)>] -> 'man'
N[NUM=sg,SEM=<\x.girl(x)>] -> 'girl'
N[NUM=sg,SEM=<\x.boy(x)>] -> 'boy'
N[NUM=sg,SEM=<\x.bone(x)>] -> 'bone'
N[NUM=sg,SEM=<\x.ankle(x)>] -> 'ankle'
N[NUM=sg,SEM=<\x.dog(x)>] -> 'dog'
N[NUM=pl,SEM=<\x.dog(x)>] -> 'dogs'

IV[NUM=sg,SEM=<\x.bark(x)>,TNS=pres] -> 'barks'
IV[NUM=pl,SEM=<\x.bark(x)>,TNS=pres] -> 'bark'
IV[NUM=sg,SEM=<\x.walk(x)>,TNS=pres] -> 'walks'
IV[NUM=pl,SEM=<\x.walk(x)>,TNS=pres] -> 'walk'
TV[NUM=sg,SEM=<\X x.X(\ y.chase(x,y))>,TNS=pres] -> 'chases'
TV[NUM=pl,SEM=<\X x.X(\ y.chase(x,y))>,TNS=pres] -> 'chase'
TV[NUM=sg,SEM=<\X x.X(\ y.see(x,y))>,TNS=pres] -> 'sees'
TV[NUM=pl,SEM=<\X x.X(\ y.see(x,y))>,TNS=pres] -> 'see'
TV[NUM=sg,SEM=<\X x.X(\ y.bite(x,y))>,TNS=pres] -> 'bites'
TV[NUM=pl,SEM=<\X x.X(\ y.bite(x,y))>,TNS=pres] -> 'bite'
DTV[NUM=sg,SEM=<\Y X x.X(\z.Y(\y.give(x,y,z)))>,TNS=pres] -> 'gives'
DTV[NUM=pl,SEM=<\Y X x.X(\z.Y(\y.give(x,y,z)))>,TNS=pres] -> 'give'

P[+to] -> 'to'
"""

Write your extension of this grammar here:

In [10]:
fcfg_string = fcfg_string_orginal + r"""
## Your answers here
# Det[???] -> ???
Det[NUM=sg,SEM=<\P Q.all x.(P(x) -> -Q(x))>] -> 'no'
Det[NUM=pl,SEM=<\P Q.all x.(P(x) -> -Q(x))>] -> 'no'
Det[NUM=sg,SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'the'

# TV[???] -> ???
TV[NUM=sg,SEM=<\X x.X(\ y.chase(x,y))>,TNS=past] -> 'chased'

# TV[???] -> ???
TV[NUM=pl,SEM=<\X x.X(\ y.chase(x,y))>,TNS=past] -> 'chased'

# CONJ -> ???
CONJ -> 'and'

# NP[???] -> NP[???] CONJ NP[???]
NP[NUM=pl, SEM=<\P.(?np1(P) & ?np2(P))>] -> NP[SEM=?np1] CONJ NP[SEM=?np2]

# N[???] -> ???
N[NUM=sg,SEM=<\x.cat(x)>] -> 'cat'

# ADJ[???] -> ???
ADJ[SEM=<\P x.(P(x) & brown(x))>] -> 'brown'
ADJ[SEM=<\P x.(P(x) & white(x))>] -> 'white'

# NP[???] -> Det[???] ADJ[???] Nom[???]
Nom[NUM=?n,SEM=<?adj(?nom)>] -> ADJ[SEM=?adj] Nom[NUM=?n,SEM=?nom]

"""

# Load `fcfg_string` as a feature grammar:
syntax = FeatureGrammar.fromstring(fcfg_string)

Run the code below without errors:

In [11]:
# comment out sentences if you couldn't find an answer for them
sentences = [
    'no man gives a bone to a dog',
    'no man gives a bone to the dog',
    'a boy and a girl chased every dog',
    'every dog chased a boy and a girl',
    'a brown cat chases a white dog',
]
for results in nltk.interpret_sents(sentences, syntax):
    for (synrep, semrep) in results:
        display(Markdown('----'))
        display_latex(semrep) # prints the SEM feature of a tree
        display_tree(synrep) # show the parse tree

----

$\forall\ x.(man(x)\ \rightarrow\ -\exists\ z_{3}.(dog(z_{3})\ \land\ \exists\ z_{2}.(bone(z_{2})\ \land\ give(x,z_{2},z_{3}))))$

The Ghostscript executable isn't found.
See http://web.mit.edu/ghostscript/www/Install.htm
If you're using a Mac, you can try installing
https://docs.brew.sh/Installation then `brew install ghostscript`


LookupError: 

If you are working with iPython which is also running behind Jupyter notebooks and you are changing grammars and want to rerun a new version without restarting you may find `nltk.data.clear_cache()` useful.

## Statement of contribution

Briefly state how many times you have met for discussions, who was present, to what degree each member contributed to the discussion and the final answers you are submitting.  

We met in person twice with all group members (Athina, Dimitrios, Yeganeh, Elisa) present. During the first session, Elisa had to leave early, so we decided to distribute the first four tasks equally. The remaining group members solved tasks 1, 2 and 4, while Elisa solved task 3 at home. During the second meeting, we solved questions 5 to 7 together. Our method was that every group member tried to solve the questions in their own copy of the jupyter notebook. Whenever we encountered a problem, everybody worked to find a solution togethers. While not everybody might have managed to come up with a solution, the discussions were pretty balanced with everyone involved.



## Marks

The assignment is marked on a 7-level scale where 4 is sufficient to complete the assignment; 5 is good solid work; 6 is excellent work, covers most of the assignment; and 7: creative work. 

This assignment has a total of 47 marks. These translate to grades as follows: 1 = 17% 2 = 34%, 3 = 50%, 4 = 67%, 5 = 75%, 6 = 84%, 7 = 92% where %s are interpreted as lower bounds to achieve that grade.